In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Plot 1D latent variable L as a function of 2D signal combinations.
For each bin in 2D signal space, fix other signals to their conditional mean.
"""

import torch
import numpy as np
import matplotlib.pyplot as plt
import os
import dill as pickle
from sklearn.preprocessing import StandardScaler
import sys

plt.close('all')

# Add parent directory to path if needed
sys.path.insert(0, 'data_expt_20_scaled_norm_bgsub')

import sys
sys.path.append('/Users/idse/repos/signaldecoding/2D_gastruloids_v5')
import fns_NN

# ============= CONFIGURATION =============
SPLIT_TYPE = 'random'
directory = 'data_expt_20_scaled_norm_bgsub'
OUTPUT_DIR = directory + '/analysis_VIB_1D_split_' + SPLIT_TYPE

# Which condition to analyze
CONDITION = 'B50'

# Mode settings
MODE_RUN = True   # If True, run analysis and save data
MODE_PLOT = True  # If True, load data and create figures

# Output directories
DATA_DIR = 'data_L_2D_signal_space'  # Main data directory for this analysis
DATA_SAVE_DIR = os.path.join(DATA_DIR, 'data')  # For computed data
FIG_DIR = os.path.join(DATA_DIR, 'figs_viridis')  # For figures (accumulates with different approaches)

# 2D plot settings
N_BINS = 30  # Number of bins per dimension
SIGNAL_PAIR_NAMES = ('Smad1', 'ERK')  # Names of signals to plot
PLOT_ALL_PAIRS = True  # If True, plot all pairwise combinations; if False, just SIGNAL_PAIR
MODE_SMOOTH = True  # If True, use smoothed interpolation; if False, use binned conditional means
MODE_ARROWS = False  # If True, display gradient arrows/streamlines; if False, hide them
DENSITY_PERCENTILE = 95  # Percentile for the grey density region (e.g., 95 means region containing 95% of data)
EXCLUDE_SIGNALS = ['AKT']  # Signals to exclude from pairwise plots (by name)
LINE_PLOT_PERCENTILES = [10, 50, 90]  # Percentiles of y-signal for line plots

# Cell fate colormap
colmap_fates = {
    'AMLC': [90/255, 166/255, 71/255, 1],
    'PGCLC': [227/255, 143/255, 52/255, 1],
    'PSLC': [211/255, 62/255, 43/255, 1],
    'meso': [140/255, 40/255, 93/255, 1],
    'pluri': [75/255, 167/255, 158/255, 1],
    'ecto': [49/255, 118/255, 181/255, 1],
    'endo': [227/255, 179/255, 61/255, 1],
    'other': [0.8, 0.8, 0.8, 1]
}

# ============= CREATE DIRECTORIES =============
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(DATA_SAVE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
print(f"Data directory: {DATA_DIR}")
print(f"Data save directory: {DATA_SAVE_DIR}")
print(f"Figure directory: {FIG_DIR}")

Data directory: data_L_2D_signal_space
Data save directory: data_L_2D_signal_space/data
Figure directory: data_L_2D_signal_space/figs_viridis


In [3]:
# ============= LOAD MODEL AND DATA =============
print("Loading model and data...")

# Load saved data
data_path = os.path.join(OUTPUT_DIR, f'latent_1D_data_{CONDITION}.pickle')
with open(data_path, 'rb') as f:
    data = pickle.load(f)

# Load model
model_path = os.path.join(OUTPUT_DIR, f'vib_1D_model_{data["train_condition"]}.pt')
checkpoint = torch.load(model_path, weights_only=False)

# Reconstruct model
config = checkpoint['model_config']
vib_1D = fns_NN.FlexibleVIB(
    input_dim=config['input_dim'],
    output_dim=config['output_dim'],
    latent_dim=config['latent_dim'],
    hidden_dim=config['hidden_dim'],
    n_layers=config['n_layers'],
    encoder_type='nonlinear',
    decoder_type='nonlinear'
)
vib_1D.load_state_dict(checkpoint['model_state_dict'])
vib_1D.eval()

scaler_X = checkpoint['scaler_X']

X_full = data['X_full']
INPUT_NAMES = data['INPUT_NAMES']
latent_1D = data['latent_1D']
markers_full = data['markers_clean']  # Use clean markers (no junk)
MARKER_NAMES = data['MARKER_NAMES']

n_signals = X_full.shape[1]
print(f"Loaded {X_full.shape[0]} samples with {n_signals} signals")
print(f"Signal names: {INPUT_NAMES}")
print(f"Cell fates: {MARKER_NAMES}")

# ============= FLIP LATENT SPACE IF NEEDED =============
# Ensure negative L corresponds to high Smad1 (first signal)
# Check correlation between L and Smad1
corr_L_Smad1 = np.corrcoef(latent_1D, X_full[:, 0])[0, 1]
print(f"\nCorrelation between L and {INPUT_NAMES[0]}: {corr_L_Smad1:.3f}")

if corr_L_Smad1 > 0:
    print(f"Flipping L so that negative L corresponds to high {INPUT_NAMES[0]}")
    FLIP_L = True
else:
    print(f"L already oriented correctly (negative L = high {INPUT_NAMES[0]})")
    FLIP_L = False

# Find signal indices by name
def find_signal_index(name, signal_names):
    """Find index of signal by name (case-insensitive partial match)."""
    for i, sn in enumerate(signal_names):
        if name.lower() in sn.lower():
            return i
    raise ValueError(f"Signal '{name}' not found in {signal_names}")

SIGNAL_PAIR = (
    find_signal_index(SIGNAL_PAIR_NAMES[0], INPUT_NAMES),
    find_signal_index(SIGNAL_PAIR_NAMES[1], INPUT_NAMES)
)
print(f"Signal pair: {SIGNAL_PAIR_NAMES} -> indices {SIGNAL_PAIR}")

Loading model and data...


FileNotFoundError: [Errno 2] No such file or directory: 'data_expt_20_scaled_norm_bgsub/analysis_VIB_1D_split_random/latent_1D_data_B50.pickle'

In [ ]:
def plot_L_2D(sig_i, sig_j, X_full, vib_1D, scaler_X, n_bins=30, flip_L=False):
    """
    Plot L as a function of two signals, with other signals set to conditional means.
    Also computes gradients of L with respect to signals at each bin.
    
    Parameters:
    -----------
    sig_i, sig_j : int
        Indices of the two signals to plot
    X_full : ndarray
        Full signal data (N x n_signals)
    vib_1D : model
        Trained VIB encoder
    scaler_X : StandardScaler
        Fitted scaler for input signals
    n_bins : int
        Number of bins per dimension
    flip_L : bool
        If True, flip sign of L and gradients
    
    Returns:
    --------
    x_centers, y_centers, L_grid, grad_grid_i, grad_grid_j
    """
    n_signals = X_full.shape[1]
    
    # Get signal ranges (use percentiles to avoid outliers)
    x_min, x_max = np.percentile(X_full[:, sig_i], [2, 98])
    y_min, y_max = np.percentile(X_full[:, sig_j], [2, 98])
    
    # Create bin edges
    x_edges = np.linspace(x_min, x_max, n_bins + 1)
    y_edges = np.linspace(y_min, y_max, n_bins + 1)
    
    # Bin centers for plotting
    x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
    y_centers = 0.5 * (y_edges[:-1] + y_edges[1:])
    
    # Initialize L grid and gradient grids
    L_grid = np.full((n_bins, n_bins), np.nan)
    grad_grid_i = np.full((n_bins, n_bins), np.nan)  # gradient wrt sig_i
    grad_grid_j = np.full((n_bins, n_bins), np.nan)  # gradient wrt sig_j
    
    # For each bin, compute conditional mean of other signals and encode
    for i in range(n_bins):
        for j in range(n_bins):
            # Find points in this bin
            mask = (
                (X_full[:, sig_i] >= x_edges[i]) & (X_full[:, sig_i] < x_edges[i+1]) &
                (X_full[:, sig_j] >= y_edges[j]) & (X_full[:, sig_j] < y_edges[j+1])
            )
            
            n_points = np.sum(mask)
            if n_points < 5:  # Skip bins with too few points
                continue
            
            # Compute mean of all signals in this bin
            mean_signals = X_full[mask].mean(axis=0)
            
            # Create input: use bin center for sig_i and sig_j, conditional mean for others
            x_input = mean_signals.copy()
            x_input[sig_i] = x_centers[i]
            x_input[sig_j] = y_centers[j]
            
            # Scale input
            x_scaled = scaler_X.transform(x_input.reshape(1, -1))
            x_torch = torch.FloatTensor(x_scaled)
            x_torch.requires_grad_(True)
            
            # Forward pass and compute gradient
            latent_mu, _ = vib_1D.encode(x_torch)
            L_val = latent_mu.detach().numpy().flatten()[0]
            
            # Compute gradient of L wrt scaled input
            grad_scaled = torch.autograd.grad(latent_mu, x_torch, retain_graph=False)[0]
            grad_scaled = grad_scaled.numpy().flatten()
            
            # Convert gradient from scaled to original space
            # dL/dx_orig = dL/dx_scaled * dx_scaled/dx_orig = dL/dx_scaled / std
            grad_orig = grad_scaled / scaler_X.scale_
            
            # Apply flip if needed
            if flip_L:
                L_val = -L_val
                grad_orig = -grad_orig
            
            L_grid[j, i] = L_val
            grad_grid_i[j, i] = grad_orig[sig_i]
            grad_grid_j[j, i] = grad_orig[sig_j]
    
    return x_centers, y_centers, L_grid, grad_grid_i, grad_grid_j


def compute_gradient_at_point(x_input, vib_1D, scaler_X, flip_L=False):
    """
    Compute gradient of L with respect to all signals at a given input point.
    
    Returns gradient in original (unscaled) signal space.
    """
    x_scaled = scaler_X.transform(x_input.reshape(1, -1))
    x_torch = torch.FloatTensor(x_scaled)
    x_torch.requires_grad_(True)
    
    latent_mu, _ = vib_1D.encode(x_torch)
    
    grad_scaled = torch.autograd.grad(latent_mu, x_torch)[0]
    grad_scaled = grad_scaled.numpy().flatten()
    
    # Convert to original space
    grad_orig = grad_scaled / scaler_X.scale_
    L_val = latent_mu.detach().numpy().flatten()[0]
    
    # Apply flip if needed
    if flip_L:
        L_val = -L_val
        grad_orig = -grad_orig
    
    return grad_orig, L_val


def plot_L_2D_smooth(sig_i, sig_j, X_full, vib_1D, scaler_X, n_bins=30, bandwidth=None, flip_L=False):
    """
    Plot L as a function of two signals using smoothed interpolation.
    
    For each grid point, uses Gaussian kernel regression to interpolate
    the holdout signals, then evaluates L on the full interpolated signal vector.
    
    Parameters:
    -----------
    sig_i, sig_j : int
        Indices of the two signals to plot
    X_full : ndarray
        Full signal data (N x n_signals)
    vib_1D : model
        Trained VIB encoder
    scaler_X : StandardScaler
        Fitted scaler for input signals
    n_bins : int
        Number of grid points per dimension
    bandwidth : float or None
        Bandwidth for Gaussian kernel (if None, uses adaptive bandwidth)
    flip_L : bool
        If True, flip sign of L and gradients
    
    Returns:
    --------
    x_grid, y_grid, L_grid, grad_grid_i, grad_grid_j
    """
    from scipy.ndimage import gaussian_filter
    
    n_signals = X_full.shape[1]
    n_samples = X_full.shape[0]
    
    # Get signal ranges (use percentiles to avoid outliers)
    x_min, x_max = np.percentile(X_full[:, sig_i], [2, 98])
    y_min, y_max = np.percentile(X_full[:, sig_j], [2, 98])
    
    # Create grid
    x_grid = np.linspace(x_min, x_max, n_bins)
    y_grid = np.linspace(y_min, y_max, n_bins)
    
    # Compute adaptive bandwidth if not provided
    if bandwidth is None:
        # Use ~5% of the data range as bandwidth
        bandwidth_x = (x_max - x_min) / 10
        bandwidth_y = (y_max - y_min) / 10
    else:
        bandwidth_x = bandwidth_y = bandwidth
    
    # Holdout signal indices
    holdout_indices = [k for k in range(n_signals) if k not in (sig_i, sig_j)]
    
    # Initialize grids
    L_grid = np.full((n_bins, n_bins), np.nan)
    grad_grid_i = np.full((n_bins, n_bins), np.nan)
    grad_grid_j = np.full((n_bins, n_bins), np.nan)
    
    # Precompute for efficiency
    X_sig_i = X_full[:, sig_i]
    X_sig_j = X_full[:, sig_j]
    X_holdout = X_full[:, holdout_indices]
    
    # For each grid point, interpolate holdout signals and compute L
    for i, x_val in enumerate(x_grid):
        for j, y_val in enumerate(y_grid):
            # Gaussian kernel weights
            dist_sq = ((X_sig_i - x_val) / bandwidth_x)**2 + ((X_sig_j - y_val) / bandwidth_y)**2
            weights = np.exp(-0.5 * dist_sq)
            weight_sum = weights.sum()
            
            # Skip if no data nearby
            if weight_sum < 1e-10:
                continue
            
            # Normalize weights
            weights = weights / weight_sum
            
            # Interpolate holdout signals (Nadaraya-Watson estimator)
            holdout_interp = (weights[:, np.newaxis] * X_holdout).sum(axis=0)
            
            # Construct full signal vector
            x_input = np.zeros(n_signals)
            x_input[sig_i] = x_val
            x_input[sig_j] = y_val
            x_input[holdout_indices] = holdout_interp
            
            # Scale input
            x_scaled = scaler_X.transform(x_input.reshape(1, -1))
            x_torch = torch.FloatTensor(x_scaled)
            x_torch.requires_grad_(True)
            
            # Forward pass and compute gradient
            latent_mu, _ = vib_1D.encode(x_torch)
            L_val = latent_mu.detach().numpy().flatten()[0]
            
            # Compute gradient of L wrt scaled input
            grad_scaled = torch.autograd.grad(latent_mu, x_torch, retain_graph=False)[0]
            grad_scaled = grad_scaled.numpy().flatten()
            
            # Convert gradient from scaled to original space
            grad_orig = grad_scaled / scaler_X.scale_
            
            # Apply flip if needed
            if flip_L:
                L_val = -L_val
                grad_orig = -grad_orig
            
            L_grid[j, i] = L_val
            grad_grid_i[j, i] = grad_orig[sig_i]
            grad_grid_j[j, i] = grad_orig[sig_j]
    
    return x_grid, y_grid, L_grid, grad_grid_i, grad_grid_j


def create_2D_plot(sig_i, sig_j, X_full, vib_1D, scaler_X, INPUT_NAMES, 
                   markers_full, MARKER_NAMES, colmap_fates, n_bins=30, flip_L=False, show_arrows=True):
    """Create a single 2D plot of L vs two signals with cell fate markers and gradient arrows."""
    from matplotlib.patches import FancyArrowPatch
    
    x_centers, y_centers, L_grid, grad_grid_i, grad_grid_j = plot_L_2D(
        sig_i, sig_j, X_full, vib_1D, scaler_X, n_bins, flip_L=flip_L
    )
    
    fig, ax = plt.subplots(figsize=(4, 3))
    
    # Set background to light grey (for NaN bins)
    ax.set_facecolor('#d0d0d0')
    
    # Remove grid lines
    ax.grid(False)
    
    # Create masked array to handle NaN values properly
    L_masked = np.ma.masked_invalid(L_grid)
    
    # Plot heatmap
    im = ax.pcolormesh(
        x_centers, y_centers, L_masked,
        cmap='viridis', shading='auto'
    )
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax, label='Latent L')
    
    # Normalize arrow lengths for visualization
    grad_mag = np.sqrt(grad_grid_i**2 + grad_grid_j**2)
    max_mag = np.nanpercentile(grad_mag, 95)
    
    # Scale factor for arrow length (in data units)
    dx = x_centers[1] - x_centers[0]
    dy = y_centers[1] - y_centers[0]
    arrow_subsample = max(1, n_bins // 15)
    arrow_scale = min(dx, dy) * arrow_subsample * 0.8
    
    # Plot small black arrows at each bin (only if show_arrows is True)
    if show_arrows:
        for i in range(0, n_bins, arrow_subsample):
            for j in range(0, n_bins, arrow_subsample):
                if np.isnan(grad_grid_i[j, i]) or np.isnan(grad_grid_j[j, i]):
                    continue
                
                # Normalize gradient direction
                mag = np.sqrt(grad_grid_i[j, i]**2 + grad_grid_j[j, i]**2)
                if mag < 1e-10:
                    continue
                
                # Arrow components (normalized, then scaled)
                arrow_dx = (grad_grid_i[j, i] / max_mag) * arrow_scale
                arrow_dy = (grad_grid_j[j, i] / max_mag) * arrow_scale
                
                ax.arrow(x_centers[i], y_centers[j], arrow_dx, arrow_dy,
                        head_width=arrow_scale*0.3, head_length=arrow_scale*0.2,
                        fc='black', ec='black', alpha=0.6, linewidth=0.5, zorder=5)
    
    # Add cell fate mean positions as scatter points with gradient arrows
    for fate_idx, fate_name in enumerate(MARKER_NAMES):
        mask = markers_full == fate_idx
        if np.sum(mask) > 0:
            # Mean signal position for this fate
            mean_signals = X_full[mask].mean(axis=0)
            mean_x = mean_signals[sig_i]
            mean_y = mean_signals[sig_j]
            
            # Get color
            color = colmap_fates.get(fate_name, colmap_fates['other'])
            
            # Compute gradient and plot arrow (only if show_arrows is True)
            if show_arrows:
                grad_orig, L_val = compute_gradient_at_point(mean_signals, vib_1D, scaler_X, flip_L=flip_L)
                
                # Plot larger arrow for fate using FancyArrowPatch
                grad_mag_fate = np.sqrt(grad_orig[sig_i]**2 + grad_orig[sig_j]**2)
                if grad_mag_fate > 1e-10:
                    fate_arrow_scale = arrow_scale * 2.5
                    arrow_dx = (grad_orig[sig_i] / max_mag) * fate_arrow_scale
                    arrow_dy = (grad_orig[sig_j] / max_mag) * fate_arrow_scale
                    
                    # Use FancyArrowPatch with Fancy style
                    arrow = FancyArrowPatch(
                        (mean_x, mean_y), (mean_x + arrow_dx, mean_y + arrow_dy),
                        arrowstyle='Fancy,head_length=8,head_width=5,tail_width=3',
                        color=color, ec='black', linewidth=0.8,
                        mutation_scale=1, zorder=15
                    )
                    ax.add_patch(arrow)
            
            # Plot scatter point
            ax.scatter(mean_x, mean_y, c=[color], s=150, edgecolors='white', 
                      linewidths=2, label=fate_name, zorder=20)
    
    # Add legend
    ax.legend(loc='upper left', fontsize=9, framealpha=0.9)
    
    # Labels
    ax.set_xlabel(INPUT_NAMES[sig_i], fontsize=11)
    ax.set_ylabel(INPUT_NAMES[sig_j], fontsize=11)
    ax.set_title(f'L({INPUT_NAMES[sig_i]}, {INPUT_NAMES[sig_j]}) with ∇L arrows\nother signals = conditional mean', fontsize=11)
    
    plt.tight_layout()
    
    return fig, ax, L_grid, grad_grid_i, grad_grid_j


def create_2D_plot_smooth(sig_i, sig_j, X_full, vib_1D, scaler_X, INPUT_NAMES, 
                          markers_full, MARKER_NAMES, colmap_fates, n_bins=30, flip_L=False, show_arrows=True):
    """Create a 2D plot of L vs two signals using smoothed interpolation and streamplot."""
    from matplotlib.patches import FancyArrowPatch
    
    x_grid, y_grid, L_grid, grad_grid_i, grad_grid_j = plot_L_2D_smooth(
        sig_i, sig_j, X_full, vib_1D, scaler_X, n_bins, flip_L=flip_L
    )
    
    fig, ax = plt.subplots(figsize=(4, 3))
    
    # Set background to light grey (for NaN regions)
    ax.set_facecolor('#d0d0d0')
    
    # Remove grid lines
    ax.grid(False)
    
    # Create masked array to handle NaN values properly
    L_masked = np.ma.masked_invalid(L_grid)
    
    # Plot heatmap using imshow for smooth appearance
    extent = [x_grid[0], x_grid[-1], y_grid[0], y_grid[-1]]
    im = ax.imshow(
        L_masked, origin='lower', extent=extent, aspect='auto',
        cmap='viridis', interpolation='bilinear'
    )
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax, label='Latent L')
    
    # Plot streamlines (only if show_arrows is True)
    if show_arrows:
        # Create meshgrid for streamplot
        X_mesh, Y_mesh = np.meshgrid(x_grid, y_grid)
        
        # Prepare velocity fields for streamplot (replace NaN with 0 for streamplot)
        U = np.nan_to_num(grad_grid_i, nan=0.0)
        V = np.nan_to_num(grad_grid_j, nan=0.0)
        
        # Compute speed for line width
        speed = np.sqrt(U**2 + V**2)
        
        # Only plot streamlines where we have valid data
        valid_mask = ~np.isnan(L_grid)
        
        if np.any(valid_mask) and np.any(speed > 1e-10):
            # Normalize speed for linewidth (range 0.3 to 2.5)
            speed_max = np.nanpercentile(speed[speed > 0], 95)
            lw = 0.3 + 2.2 * (speed / speed_max)
            lw = np.clip(lw, 0.3, 2.5)
            
            strm = ax.streamplot(
                X_mesh, Y_mesh, U, V,
                color='#404040', linewidth=lw,
                density=1.5, arrowsize=0.8, arrowstyle='->',
                minlength=0.2
            )
    
    # Add cell fate mean positions as scatter points with gradient arrows
    # Compute max gradient magnitude for arrow scaling
    grad_mag = np.sqrt(grad_grid_i**2 + grad_grid_j**2)
    max_mag = np.nanpercentile(grad_mag, 95)
    dx = x_grid[1] - x_grid[0]
    dy = y_grid[1] - y_grid[0]
    arrow_scale = min(dx, dy) * 3
    
    for fate_idx, fate_name in enumerate(MARKER_NAMES):
        mask = markers_full == fate_idx
        if np.sum(mask) > 0:
            # Mean signal position for this fate
            mean_signals = X_full[mask].mean(axis=0)
            mean_x = mean_signals[sig_i]
            mean_y = mean_signals[sig_j]
            
            # Get color
            color = colmap_fates.get(fate_name, colmap_fates['other'])
            
            # Compute gradient and plot arrow (only if show_arrows is True)
            if show_arrows:
                grad_orig, L_val = compute_gradient_at_point(mean_signals, vib_1D, scaler_X, flip_L=flip_L)
                
                # Plot larger arrow for fate using FancyArrowPatch
                grad_mag_fate = np.sqrt(grad_orig[sig_i]**2 + grad_orig[sig_j]**2)
                if grad_mag_fate > 1e-10 and max_mag > 1e-10:
                    fate_arrow_scale = arrow_scale * 2.5
                    arrow_dx = (grad_orig[sig_i] / max_mag) * fate_arrow_scale
                    arrow_dy = (grad_orig[sig_j] / max_mag) * fate_arrow_scale
                    
                    # Use FancyArrowPatch with Fancy style
                    arrow = FancyArrowPatch(
                        (mean_x, mean_y), (mean_x + arrow_dx, mean_y + arrow_dy),
                        arrowstyle='Fancy,head_length=8,head_width=5,tail_width=3',
                        color=color, ec='black', linewidth=0.8,
                        mutation_scale=1, zorder=15
                    )
                    ax.add_patch(arrow)
            
            # Plot scatter point
            ax.scatter(mean_x, mean_y, c=[color], s=150, edgecolors='white', 
                      linewidths=2, label=fate_name, zorder=20)
    
    # Add legend
    ax.legend(loc='upper left', fontsize=9, framealpha=0.9)
    
    # Labels
    ax.set_xlabel(INPUT_NAMES[sig_i], fontsize=11)
    ax.set_ylabel(INPUT_NAMES[sig_j], fontsize=11)
    ax.set_title(f'L({INPUT_NAMES[sig_i]}, {INPUT_NAMES[sig_j]}) with ∇L streamlines\nsmoothed interpolation', fontsize=11)
    
    plt.tight_layout()
    
    return fig, ax, L_grid, grad_grid_i, grad_grid_j

In [ ]:
# ============= CREATE PLOTS =============
print("\n" + "="*60)
print("CREATING 2D LATENT PLOTS")
print("="*60)

# Use configured figure directory
fig_dir = FIG_DIR

n_signals = X_full.shape[1]

# Find indices of signals to exclude
exclude_indices = []
for excl_name in EXCLUDE_SIGNALS:
    for idx, sig_name in enumerate(INPUT_NAMES):
        if excl_name.lower() in sig_name.lower():
            exclude_indices.append(idx)
            print(f"Excluding signal: {sig_name} (index {idx})")

# Get indices of signals to include
include_indices = [i for i in range(n_signals) if i not in exclude_indices]
n_included = len(include_indices)
print(f"Included signals ({n_included}): {[INPUT_NAMES[i] for i in include_indices]}")

if MODE_RUN:
    print("\n--- MODE_RUN: Computing grids ---")
    
    if PLOT_ALL_PAIRS:
        # Collect all pairs first (i < j to avoid double counting), excluding specified signals
        pairs = [(i, j) for i in range(n_signals) for j in range(i+1, n_signals)
                 if i not in exclude_indices and j not in exclude_indices]
        
        n_pairs = len(pairs)
        print(f"\nGenerating {n_pairs} pairwise plots...")
        print(f"Mode: {'Smoothed interpolation' if MODE_SMOOTH else 'Binned conditional means'}")
        print(f"Arrows: {MODE_ARROWS}")
        print(f"L flip: {FLIP_L}")
        print(f"Density percentile: {DENSITY_PERCENTILE}%")
        
        # First pass: compute axis limits using binned approach (consistent for both modes)
        print("  Computing axis limits and grids...")
        all_L_grids = {}
        axis_limits = {}  # Store (x_min, x_max, y_min, y_max) for each pair
        global_L_min = np.inf
        global_L_max = -np.inf
        
        for (i, j) in pairs:
            # Always compute binned version first for consistent axis limits
            x_grid_binned, y_grid_binned, L_grid_binned, grad_i_binned, grad_j_binned = plot_L_2D(
                i, j, X_full, vib_1D, scaler_X, N_BINS, flip_L=FLIP_L
            )
            
            # Store axis limits from binned version
            axis_limits[(i, j)] = {
                'x_min': x_grid_binned[0], 'x_max': x_grid_binned[-1],
                'y_min': y_grid_binned[0], 'y_max': y_grid_binned[-1]
            }
            
            # Update global L limits from binned version
            valid_L = L_grid_binned[~np.isnan(L_grid_binned)]
            if len(valid_L) > 0:
                global_L_min = min(global_L_min, np.min(valid_L))
                global_L_max = max(global_L_max, np.max(valid_L))
            
            # Now compute the version we'll actually use for plotting
            if MODE_SMOOTH:
                x_grid, y_grid, L_grid, grad_grid_i, grad_grid_j = plot_L_2D_smooth(
                    i, j, X_full, vib_1D, scaler_X, N_BINS, flip_L=FLIP_L
                )
            else:
                x_grid, y_grid, L_grid, grad_grid_i, grad_grid_j = (
                    x_grid_binned, y_grid_binned, L_grid_binned, grad_i_binned, grad_j_binned
                )
            
            all_L_grids[(i, j)] = {
                'x_grid': x_grid,
                'y_grid': y_grid,
                'L_grid': L_grid,
                'grad_grid_i': grad_grid_i,
                'grad_grid_j': grad_grid_j
            }
        
        print(f"  Global L range (computed): [{global_L_min:.3f}, {global_L_max:.3f}]")
        
        # Use fixed color limits
        global_L_min = -2.0
        global_L_max = 2.0
        print(f"  Using fixed L range: [{global_L_min:.1f}, {global_L_max:.1f}]")
    
    # Dynamic grid layout based on number of pairs
    n_cols = min(5, n_pairs)  # Max 5 columns
    n_rows = int(np.ceil(n_pairs / n_cols))
    
    fig_all, axes_all = plt.subplots(n_rows, n_cols, figsize=(2.8*n_cols, 2.5*n_rows))
    if n_pairs == 1:
        axes_flat = [axes_all]
    elif n_rows == 1:
        axes_flat = list(axes_all)
    else:
        axes_flat = axes_all.flatten()
    
    # Import for density contour
    from scipy.stats import gaussian_kde
    from matplotlib.patches import FancyArrowPatch
    
    # Second pass: plot with consistent color limits
    for pair_idx, (i, j) in enumerate(pairs):
        print(f"  Plotting {INPUT_NAMES[i]} vs {INPUT_NAMES[j]}...")
        
        grids = all_L_grids[(i, j)]
        x_grid = grids['x_grid']
        y_grid = grids['y_grid']
        L_grid = grids['L_grid']
        grad_grid_i = grids['grad_grid_i']
        grad_grid_j = grids['grad_grid_j']
        
        # Get consistent axis limits from binned version
        ax_lim = axis_limits[(i, j)]
        
        # Plot in grid
        ax = axes_flat[pair_idx]
        ax.set_facecolor('#d0d0d0')
        ax.grid(False)
        L_masked = np.ma.masked_invalid(L_grid)
        
        if MODE_SMOOTH:
            # Use imshow for smooth appearance
            extent = [ax_lim['x_min'], ax_lim['x_max'], ax_lim['y_min'], ax_lim['y_max']]
            im = ax.imshow(
                L_masked, origin='lower', extent=extent, aspect='auto',
                cmap='viridis', interpolation='bilinear',
                vmin=global_L_min, vmax=global_L_max
            )
            
            # Streamplot for vector field (only if MODE_ARROWS is True)
            if MODE_ARROWS:
                X_mesh, Y_mesh = np.meshgrid(x_grid, y_grid)
                U = np.nan_to_num(grad_grid_i, nan=0.0)
                V = np.nan_to_num(grad_grid_j, nan=0.0)
                speed = np.sqrt(U**2 + V**2)
                
                if np.any(speed > 1e-10):
                    # Normalize speed for linewidth (range 0.3 to 2.0)
                    speed_max = np.nanpercentile(speed[speed > 0], 95)
                    lw = 0.3 + 1.7 * (speed / speed_max)
                    lw = np.clip(lw, 0.3, 2.0)
                    
                    ax.streamplot(
                        X_mesh, Y_mesh, U, V,
                        color='#404040', linewidth=lw,
                        density=1.0, arrowsize=0.5, arrowstyle='->',
                        minlength=0.1
                    )
        else:
            im = ax.pcolormesh(
                x_grid, y_grid, L_masked,
                cmap='viridis', shading='auto',
                vmin=global_L_min, vmax=global_L_max
            )
            
            # Add small black arrows (only if MODE_ARROWS is True)
            if MODE_ARROWS:
                arrow_subsample = max(1, N_BINS // 12)
                grad_mag = np.sqrt(grad_grid_i**2 + grad_grid_j**2)
                max_mag = np.nanpercentile(grad_mag, 95)
                if max_mag > 1e-10:
                    dx = x_grid[1] - x_grid[0]
                    dy = y_grid[1] - y_grid[0]
                    arrow_scale = min(dx, dy) * arrow_subsample * 0.7
                
                for ii in range(0, N_BINS, arrow_subsample):
                    for jj in range(0, N_BINS, arrow_subsample):
                        if np.isnan(grad_grid_i[jj, ii]) or np.isnan(grad_grid_j[jj, ii]):
                            continue
                        mag = np.sqrt(grad_grid_i[jj, ii]**2 + grad_grid_j[jj, ii]**2)
                        if mag < 1e-10:
                            continue
                        arrow_dx = (grad_grid_i[jj, ii] / max_mag) * arrow_scale
                        arrow_dy = (grad_grid_j[jj, ii] / max_mag) * arrow_scale
                        ax.arrow(x_grid[ii], y_grid[jj], arrow_dx, arrow_dy,
                                head_width=arrow_scale*0.3, head_length=arrow_scale*0.2,
                                fc='black', ec='black', alpha=0.5, linewidth=0.4, zorder=5)
        
        # Set consistent axis limits
        ax.set_xlim(ax_lim['x_min'], ax_lim['x_max'])
        ax.set_ylim(ax_lim['y_min'], ax_lim['y_max'])
        
        # Add density contour for DENSITY_PERCENTILE region
        x_data = X_full[:, i]
        y_data = X_full[:, j]
        
        # Filter to data within plot range
        in_range = ((x_data >= ax_lim['x_min']) & (x_data <= ax_lim['x_max']) & 
                   (y_data >= ax_lim['y_min']) & (y_data <= ax_lim['y_max']))
        x_data_filt = x_data[in_range]
        y_data_filt = y_data[in_range]
        
        if len(x_data_filt) > 100:
            try:
                kde = gaussian_kde(np.vstack([x_data_filt, y_data_filt]))
                X_kde, Y_kde = np.meshgrid(
                    np.linspace(ax_lim['x_min'], ax_lim['x_max'], N_BINS),
                    np.linspace(ax_lim['y_min'], ax_lim['y_max'], N_BINS)
                )
                positions = np.vstack([X_kde.ravel(), Y_kde.ravel()])
                Z_kde = kde(positions).reshape(X_kde.shape)
                
                # Find threshold for DENSITY_PERCENTILE region
                kde_vals_at_data = kde(np.vstack([x_data_filt, y_data_filt]))
                threshold = np.percentile(kde_vals_at_data, 100 - DENSITY_PERCENTILE)
                
                # Plot contour as dotted white line
                ax.contour(X_kde, Y_kde, Z_kde, levels=[threshold], 
                          colors=['white'], linewidths=1.0, linestyles='dotted', zorder=5)
            except:
                pass  # Skip if KDE fails
        
        # Set axis labels (signal names) with small font - no title
        ax.set_xlabel(INPUT_NAMES[i], fontsize=7)
        ax.set_ylabel(INPUT_NAMES[j], fontsize=7)
        
        # Set min/max tick labels with small font
        ax.set_xticks([ax_lim['x_min'], ax_lim['x_max']])
        ax.set_xticklabels([f'{ax_lim["x_min"]:.1f}', f'{ax_lim["x_max"]:.1f}'], fontsize=6)
        ax.set_yticks([ax_lim['y_min'], ax_lim['y_max']])
        ax.set_yticklabels([f'{ax_lim["y_min"]:.1f}', f'{ax_lim["y_max"]:.1f}'], fontsize=6)
        
        # Add cell fate markers with fancy arrows using FancyArrowPatch
        grad_mag = np.sqrt(grad_grid_i**2 + grad_grid_j**2)
        max_mag = np.nanpercentile(grad_mag, 95)
        dx = x_grid[1] - x_grid[0]
        dy = y_grid[1] - y_grid[0]
        arrow_scale = min(dx, dy) * 2
        
        for fate_idx, fate_name in enumerate(MARKER_NAMES):
            mask = markers_full == fate_idx
            if np.sum(mask) > 0:
                mean_signals = X_full[mask].mean(axis=0)
                mean_x = mean_signals[i]
                mean_y = mean_signals[j]
                color = colmap_fates.get(fate_name, colmap_fates['other'])
                
                # Compute gradient at fate centroid and draw arrow (only if MODE_ARROWS is True)
                if MODE_ARROWS:
                    grad_orig, L_val = compute_gradient_at_point(mean_signals, vib_1D, scaler_X, flip_L=FLIP_L)
                    if max_mag > 1e-10:
                        fate_arrow_scale = arrow_scale * 2.5
                        arrow_dx = (grad_orig[i] / max_mag) * fate_arrow_scale
                        arrow_dy = (grad_orig[j] / max_mag) * fate_arrow_scale
                        
                        # Use FancyArrowPatch with Fancy style
                        arrow = FancyArrowPatch(
                            (mean_x, mean_y), (mean_x + arrow_dx, mean_y + arrow_dy),
                            arrowstyle='Fancy,head_length=6,head_width=4,tail_width=2',
                            color=color, ec='black', linewidth=0.5,
                            mutation_scale=1, zorder=15
                        )
                        ax.add_patch(arrow)
                
                # Only add label on first panel for legend
                label = fate_name if pair_idx == 0 else None
                ax.scatter(mean_x, mean_y, c=[color], s=60, edgecolors='white', 
                          linewidths=1.5, zorder=20, label=label)
    
    # Hide unused axes
    for idx in range(len(pairs), len(axes_flat)):
        axes_flat[idx].set_visible(False)
    
    # Add single colorbar for all panels
    fig_all.subplots_adjust(right=0.88, top=0.88)
    cbar_ax = fig_all.add_axes([0.90, 0.15, 0.015, 0.55])
    sm = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(vmin=global_L_min, vmax=global_L_max))
    cbar = fig_all.colorbar(sm, cax=cbar_ax)
    cbar.set_label('Latent L', fontsize=10)
    
    # Add legend for cell fates at top right above colorbar
    legend_handles = []
    for fate_name in MARKER_NAMES:
        color = colmap_fates.get(fate_name, colmap_fates['other'])
        handle = plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color,
                           markersize=8, markeredgecolor='white', markeredgewidth=1, label=fate_name)
        legend_handles.append(handle)
    
    fig_all.legend(handles=legend_handles, loc='upper right', bbox_to_anchor=(0.99, 0.99),
                   fontsize=8, framealpha=0.9, ncol=2)
    
    plt.tight_layout(rect=[0, 0, 0.88, 0.88])
    
    # Save grid figure
    suffix = '_smooth' if MODE_SMOOTH else ''
    fig_path = os.path.join(fig_dir, f'L_2D_all_pairs_{CONDITION}{suffix}.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"\n✓ Saved all-pairs figure to: {fig_path}")

else:
    # Just plot the single specified pair
    sig_i, sig_j = SIGNAL_PAIR
    print(f"\nPlotting {INPUT_NAMES[sig_i]} vs {INPUT_NAMES[sig_j]}...")
    print(f"Mode: {'Smoothed interpolation' if MODE_SMOOTH else 'Binned conditional means'}")
    print(f"Arrows: {MODE_ARROWS}")
    print(f"L flip: {FLIP_L}")
    
    if MODE_SMOOTH:
        fig, ax, L_grid, grad_grid_i, grad_grid_j = create_2D_plot_smooth(
            sig_i, sig_j, X_full, vib_1D, scaler_X, INPUT_NAMES,
            markers_full, MARKER_NAMES, colmap_fates, N_BINS, flip_L=FLIP_L, show_arrows=MODE_ARROWS
        )
        fig_path = os.path.join(fig_dir, f'L_2D_{INPUT_NAMES[sig_i]}_{INPUT_NAMES[sig_j]}_{CONDITION}_smooth.png')
    else:
        fig, ax, L_grid, grad_grid_i, grad_grid_j = create_2D_plot(
            sig_i, sig_j, X_full, vib_1D, scaler_X, INPUT_NAMES,
            markers_full, MARKER_NAMES, colmap_fates, N_BINS, flip_L=FLIP_L, show_arrows=MODE_ARROWS
        )
        fig_path = os.path.join(fig_dir, f'L_2D_{INPUT_NAMES[sig_i]}_{INPUT_NAMES[sig_j]}_{CONDITION}.png')
    
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {fig_path}")
    
    # Save grid data
    all_L_grids = {(sig_i, sig_j): {
        'L_grid': L_grid,
        'grad_grid_i': grad_grid_i,
        'grad_grid_j': grad_grid_j
    }}

# Save computed grids
grid_path = os.path.join(DATA_SAVE_DIR, f'L_2D_grids_{CONDITION}.pickle')
with open(grid_path, 'wb') as f:
    pickle.dump({
        'all_L_grids': all_L_grids,
        'INPUT_NAMES': INPUT_NAMES,
        'N_BINS': N_BINS,
        'condition': CONDITION,
        'FLIP_L': FLIP_L,
        'include_indices': include_indices,
        'exclude_indices': exclude_indices
    }, f)
print(f"✓ Saved grid data to: {grid_path}")

In [ ]:
# ============= EXTRA FIGURE: SIGNALS VS L AND FATE PROBABILITY VS L =============
print("\n" + "="*60)
print("CREATING SIGNALS VS L AND FATE PROBABILITY PLOTS")
print("="*60)

# Compute L for all data points (with flip if needed)
X_full_scaled = scaler_X.transform(X_full)
X_full_torch = torch.FloatTensor(X_full_scaled)

with torch.no_grad():
    L_all, _ = vib_1D.encode(X_full_torch)
    L_all = L_all.numpy().flatten()

if FLIP_L:
    L_all = -L_all

# Create figure with 2 panels
fig_summary, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: All 7 signals vs L
print("  Plotting signals vs L...")

# Create color palette for signals
signal_colors = plt.cm.tab10(np.linspace(0, 1, n_signals))

for sig_idx in range(n_signals):
    ax1.scatter(L_all, X_full[:, sig_idx], c=[signal_colors[sig_idx]], 
               s=5, alpha=0.3, label=INPUT_NAMES[sig_idx])

ax1.set_xlabel('Latent L', fontsize=11)
ax1.set_ylabel('Signal intensity', fontsize=11)
ax1.set_title('All signals vs Latent L', fontsize=12)
ax1.legend(loc='upper right', fontsize=9, markerscale=3, framealpha=0.9)
ax1.axvline(0, color='grey', linestyle='--', linewidth=0.5, alpha=0.5)

# Panel 2: Probability of each cell fate vs L (binned)
print("  Plotting fate probabilities vs L...")

# Bin L values
n_L_bins = 30
L_bin_edges = np.linspace(np.percentile(L_all, 2), np.percentile(L_all, 98), n_L_bins + 1)
L_bin_centers = 0.5 * (L_bin_edges[:-1] + L_bin_edges[1:])

# Compute fate probabilities in each bin
fate_probs = np.zeros((n_L_bins, len(MARKER_NAMES)))

for bin_idx in range(n_L_bins):
    mask = (L_all >= L_bin_edges[bin_idx]) & (L_all < L_bin_edges[bin_idx + 1])
    n_in_bin = np.sum(mask)
    if n_in_bin > 0:
        for fate_idx in range(len(MARKER_NAMES)):
            fate_probs[bin_idx, fate_idx] = np.sum(markers_full[mask] == fate_idx) / n_in_bin

# Plot stacked or line plot
for fate_idx, fate_name in enumerate(MARKER_NAMES):
    color = colmap_fates.get(fate_name, colmap_fates['other'])
    ax2.plot(L_bin_centers, fate_probs[:, fate_idx], color=color, 
            linewidth=2, label=fate_name)
    ax2.fill_between(L_bin_centers, 0, fate_probs[:, fate_idx], 
                     color=color, alpha=0.2)

ax2.set_xlabel('Latent L', fontsize=11)
ax2.set_ylabel('Probability (fraction of cells)', fontsize=11)
ax2.set_title('Cell fate probability vs Latent L', fontsize=12)
ax2.legend(loc='upper right', fontsize=9, framealpha=0.9)
ax2.set_ylim(0, 1)
ax2.axvline(0, color='grey', linestyle='--', linewidth=0.5, alpha=0.5)

plt.tight_layout()

# Save figure
fig_path = os.path.join(fig_dir, f'signals_and_fates_vs_L_{CONDITION}.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"✓ Saved: {fig_path}")

In [4]:
# ============= LINE PLOT: L VS SIGNAL FOR DIFFERENT PERCENTILES OF OTHER SIGNAL =============
print("\n" + "="*60)
print("CREATING LINE PLOTS: L VS SIGNAL FOR PERCENTILES OF Y-SIGNAL")
print("="*60)

from scipy.stats import gaussian_kde

def compute_L_along_line(sig_i, sig_j, sig_j_value, X_full, vib_1D, scaler_X, 
                         n_points=100, bandwidth=None, flip_L=False):
    """
    Compute L as a function of sig_i, with sig_j held fixed, and other signals
    set to their conditional mean given (sig_i, sig_j).
    
    Returns x_vals, L_vals, mask_valid (True where within 99th percentile of joint distribution)
    """
    n_signals = X_full.shape[1]
    
    # Get signal range for sig_i
    x_min, x_max = np.percentile(X_full[:, sig_i], [2, 98])
    x_vals = np.linspace(x_min, x_max, n_points)
    
    # Compute adaptive bandwidth if not provided
    if bandwidth is None:
        bandwidth_x = (x_max - x_min) / 10
        bandwidth_y = (np.percentile(X_full[:, sig_j], 98) - np.percentile(X_full[:, sig_j], 2)) / 10
    else:
        bandwidth_x = bandwidth_y = bandwidth
    
    # Holdout signal indices (all except sig_i and sig_j)
    holdout_indices = [k for k in range(n_signals) if k not in (sig_i, sig_j)]
    
    # Precompute for efficiency
    X_sig_i = X_full[:, sig_i]
    X_sig_j = X_full[:, sig_j]
    X_holdout = X_full[:, holdout_indices]
    
    # Compute 2D KDE for masking to 99th percentile region
    try:
        kde_2d = gaussian_kde(np.vstack([X_sig_i, X_sig_j]))
        kde_vals_at_data = kde_2d(np.vstack([X_sig_i, X_sig_j]))
        threshold_99 = np.percentile(kde_vals_at_data, 1)  # 1st percentile = 99th percentile region boundary
    except:
        threshold_99 = 0  # If KDE fails, don't mask
        kde_2d = None
    
    L_vals = np.full(n_points, np.nan)
    mask_valid = np.zeros(n_points, dtype=bool)
    
    for idx, x_val in enumerate(x_vals):
        # Check if (x_val, sig_j_value) is within 99th percentile region
        if kde_2d is not None:
            kde_val = kde_2d([[x_val], [sig_j_value]])[0]
            if kde_val < threshold_99:
                continue  # Outside 99th percentile region
        
        mask_valid[idx] = True
        
        # Gaussian kernel weights for conditional mean
        dist_sq = ((X_sig_i - x_val) / bandwidth_x)**2 + ((X_sig_j - sig_j_value) / bandwidth_y)**2
        weights = np.exp(-0.5 * dist_sq)
        weight_sum = weights.sum()
        
        if weight_sum < 1e-10:
            mask_valid[idx] = False
            continue
        
        # Normalize weights
        weights = weights / weight_sum
        
        # Interpolate holdout signals (Nadaraya-Watson estimator)
        holdout_interp = (weights[:, np.newaxis] * X_holdout).sum(axis=0)
        
        # Construct full signal vector
        x_input = np.zeros(n_signals)
        x_input[sig_i] = x_val
        x_input[sig_j] = sig_j_value
        x_input[holdout_indices] = holdout_interp
        
        # Scale input and compute L
        x_scaled = scaler_X.transform(x_input.reshape(1, -1))
        x_torch = torch.FloatTensor(x_scaled)
        
        with torch.no_grad():
            latent_mu, _ = vib_1D.encode(x_torch)
            L_val = latent_mu.numpy().flatten()[0]
        
        if flip_L:
            L_val = -L_val
        
        L_vals[idx] = L_val
    
    return x_vals, L_vals, mask_valid


# exclude_indices already computed above

# Get list of included signal indices (6 signals excluding AKT)
# include_indices already computed above
n_included = len(include_indices)
print(f"Creating 6x6 line plot grid for {n_included} signals...")
print(f"Signals: {[INPUT_NAMES[i] for i in include_indices]}")
print(f"Percentiles of y-signal: {LINE_PLOT_PERCENTILES}")

# Path for saving/loading line data
line_data_path = os.path.join(DATA_SAVE_DIR, f'L_lines_6x6_{CONDITION}.pickle')

if MODE_RUN:
    print("\n--- MODE_RUN: Computing L lines ---")
    
    # Global y-limits for L
    all_L_vals = []
    
    # Compute all lines and collect L values
    all_line_data = {}
    
    for row_idx, sig_i in enumerate(include_indices):
        for col_idx, sig_j in enumerate(include_indices):
            # Skip diagonal (can't vary sig_i at fixed sig_i)
            if sig_i == sig_j:
                continue
                
            print(f"  Computing L lines: {INPUT_NAMES[sig_i]} vs {INPUT_NAMES[sig_j]} percentiles...")
            
            # Get percentile values for sig_j
            sig_j_percentile_values = [np.percentile(X_full[:, sig_j], p) for p in LINE_PLOT_PERCENTILES]
            
            line_data = []
            for pct_idx, (pct, sig_j_val) in enumerate(zip(LINE_PLOT_PERCENTILES, sig_j_percentile_values)):
                x_vals, L_vals, mask_valid = compute_L_along_line(
                    sig_i, sig_j, sig_j_val, X_full, vib_1D, scaler_X, 
                    n_points=100, flip_L=FLIP_L
                )
                
                # Collect for global y-limits
                valid_L = L_vals[mask_valid & ~np.isnan(L_vals)]
                if len(valid_L) > 0:
                    all_L_vals.extend(valid_L)
                
                line_data.append((x_vals, L_vals, mask_valid, pct))
            
            all_line_data[(row_idx, col_idx)] = line_data
    
    # Compute global y-limits
    if len(all_L_vals) > 0:
        L_min, L_max = np.percentile(all_L_vals, [2, 98])
        L_margin = (L_max - L_min) * 0.1
        y_lim = (L_min - L_margin, L_max + L_margin)
    else:
        y_lim = (-2, 2)  # Default
    
    # Save computed data
    with open(line_data_path, 'wb') as f:
        pickle.dump({
            'all_line_data': all_line_data,
            'y_lim': y_lim,
            'include_indices': include_indices,
            'INPUT_NAMES': INPUT_NAMES,
            'LINE_PLOT_PERCENTILES': LINE_PLOT_PERCENTILES,
            'CONDITION': CONDITION
        }, f)
    print(f"✓ Saved L lines data to: {line_data_path}")

if MODE_PLOT:
    print("\n--- MODE_PLOT: Creating L lines figure ---")
    
    # Load computed data
    with open(line_data_path, 'rb') as f:
        line_data_saved = pickle.load(f)
    
    all_line_data = line_data_saved['all_line_data']
    y_lim = line_data_saved['y_lim']
    
    # Color palette for different percentile lines
    percentile_colors = ['#1f77b4', '#2ca02c', '#d62728']  # blue, green, red for 10th, 50th, 90th
    
    # Create 6x6 figure: rows = x-axis signal, columns = y-signal held at percentiles
    fig_lines, axes_lines = plt.subplots(n_included, n_included, figsize=(2.5*n_included, 2.2*n_included))
    
    # Plot
    for row_idx, sig_i in enumerate(include_indices):
        for col_idx, sig_j in enumerate(include_indices):
            ax = axes_lines[row_idx, col_idx]
            
            # Skip diagonal (hide the panel)
            if sig_i == sig_j:
                ax.set_visible(False)
                continue
            
            ax.grid(False)
            
            line_data = all_line_data[(row_idx, col_idx)]
            
            for pct_idx, (x_vals, L_vals, mask_valid, pct) in enumerate(line_data):
                color = percentile_colors[pct_idx]
                label = f'{pct}%'
                
                # Create segments for valid regions
                L_plot = L_vals.copy()
                L_plot[~mask_valid] = np.nan
                
                ax.plot(x_vals, L_plot, color=color, linewidth=1.5, label=label)
            
            # Set y-limits
            if y_lim is not None:
                ax.set_ylim(y_lim)
            
            # Column titles (top row only)
            if row_idx == 0:
                ax.set_title(f'{INPUT_NAMES[sig_j]} held', fontsize=8)
            
            # No x-axis labels (removed as requested)
            ax.set_xticklabels([])
            
            # Y-axis labels (leftmost column: show row signal as ylabel)
            if col_idx == 0:
                ax.set_ylabel(f'L (x={INPUT_NAMES[sig_i]})', fontsize=7)
            else:
                ax.set_yticklabels([])
            
            ax.tick_params(axis='both', labelsize=6)
            
            # Add legend to every panel
            ax.legend(fontsize=5, loc='best', framealpha=0.9, title=INPUT_NAMES[sig_j])
    
    plt.tight_layout()
    
    # Save line plot figure
    fig_path = os.path.join(fig_dir, f'L_vs_signal_lines_6x6_{CONDITION}.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved 6x6 line plots to: {fig_path}")

plt.show()

print("\n" + "="*60)
print("ANALYSIS COMPLETE")
print("="*60)